<a href="https://colab.research.google.com/github/Abiexperiments/Sistemas-Operativos-Garcia-Rodriguez-Abigail/blob/main/Abigail_Garcia_Practica1_Conceptos_SO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Exploración de la Estructura del SO y Recursos del Sistema
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad I:** Conceptos y Evolución de los Sistemas Operativos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Identificar la estructura física y lógica de la máquina virtual Linux subyacente en Google Colab.
2. Analizar el rol del Kernel en la administración de recursos hardware (CPU, Memoria, E/S).
3. Ejecutar y comprender llamadas al sistema (Syscalls) y la arquitectura en modo Kernel/Usuario.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede a tu entorno de **NotebookLM del Curso**, abre la sección de la *Unidad I* y consulta:
---

- *¿Qué es el Kernel de Linux y cómo abstrae el hardware?*  
**RESPUESTA:**El Kernel de Linux (o núcleo) es la pieza de software más fundamental del sistema operativo, la cual se ejecuta directamente sobre el hardware físico en el denominado modo kernel (o modo supervisor).

Abstracción de Dispositivos: El modelo "Todo es un archivo" y los Controladores
Aislamiento a través de controladores: El hardware se compone de dispositivos muy diversos con comandos y registros específicos. El kernel de Linux implementa controladores de dispositivos (device drivers) para aislar el resto del sistema de estas peculiaridades.

Archivos especiales (/dev): Siguiendo la filosofía de los sistemas Unix, Linux abstrae el acceso al hardware representando a los dispositivos de E/S como archivos especiales en la jerarquía del sistema de archivos, usualmente dentro del directorio /dev.

Clasificación de dispositivos: Para estructurar estas abstracciones de forma de manera ordenada, el kernel clasifica todos los controladores en dos categorías principales
Dispositivos de caracteres: Manejan flujos de caracteres de forma lineal y secuencial sin soporte para acceso aleatorio (como teclados o terminales). El flujo se puede procesar a través de una "disciplina de línea" para aplicar formato y edición local antes de ser enviado a la aplicación.
Dispositivos de bloques: Permiten accesos aleatorios y búsquedas (seeks) en bloques de tamaño fijo (como discos duros o CD-ROMs).

El Sistema de Archivos Virtual (VFS)
Para evitar que las aplicaciones tengan que lidiar con los detalles específicos de cada dispositivo de almacenamiento y formato de disco, Linux incluye un nivel llamado Sistema de Archivos Virtual (VFS). El VFS define un modelo de archivos común que puede representar cualquier comportamiento de almacenamiento concebible

Abstracción del Procesador (CPU)
Para abstraer el procesador, el kernel implementa el concepto de procesos e hilos, los cuales dividen una sola CPU física en múltiples CPU virtuales que ejecutan concurrentemente.

Abstracción de la Memoria (Memoria Virtual)
El kernel gestiona la memoria física a través de la memoria virtual. Con el apoyo de la unidad de gestión de memoria de hardware (MMU), el kernel de Linux traduce de forma dinámica las direcciones virtuales (o lógicas) generadas por los procesos en direcciones físicas en la RAM.

Abstracción de Red
El hardware de red (tarjetas de interfaz) y los protocolos de comunicación se abstraen a través de la interfaz de sockets, modelada de acuerdo con la convención de Berkeley.En Linux, un socket se implementa internamente como un archivo especial mediante el sistema de archivos sockfs se traduce para el programador en una API donde puede escribir y leer flujos de bytes como si se tratara de un archivo ordinario, abstrayendo toda la complejidad del direccionamiento y la encapsulación de las tramas físicas.

- *¿Qué función cumple la jerarquía de memoria en un SO moderno?*  
**RESPUESTA:**  
1. Balancear costo, capacidad y velocidad
Las tecnologías de memoria con menores tiempos de acceso (como los registros y las memorias caché SRAM) son muy costosas por bit y tienen una capacidad limitada. Por el contrario, las memorias masivas (como discos y cintas) ofrecen grandes capacidades a muy bajo costo, pero con tiempos de acceso significativamente más lentos. La jerarquía organiza los componentes en niveles organizados (Registros $\rightarrow$ Caché $\rightarrow$ Memoria Principal/RAM $\rightarrow$ Disco / Almacenamiento Secundario) para lograr un rendimiento global óptimo a un costo razonable.

 2. Mitigar la brecha de velocidad entre CPU y memoria

Existe una disparidad de rendimiento histórica (asociada al cuello de botella de Von Neumann) entre la altísima velocidad de procesamiento de la CPU y la velocidad comparativamente lenta de la memoria principal. La jerarquía permite abastecer al procesador a velocidades cercanas a su ritmo de ejecución, evitando ciclos de espera inactivos

3. Aprovechar el Principio de Localidad de Referencia

La jerarquía es viable gracias a que los programas exhiben proximidad o localidad de referencia durante su ejecución.

**Localidad temporal:** Existe una alta probabilidad de volver a acceder en un futuro cercano a las instrucciones o datos utilizados recientemente (por ejemplo, en bucles o subrutinas)

**Localidad espacial:** Existe una alta probabilidad de acceder a direcciones contiguas o cercanas a la posición actual (por ejemplo, al recorrer arreglos o código secuencial)

4. Administrar la migración de datos y la abstracción del almacenamiento (Rol del SO)

Mientras que los niveles superiores (caché L1/L2/L3) son gestionados directamente por el hardware y la MMU, el **sistema operativo** es responsable de administrar la interacción entre la memoria principal y la secundaria.
* **Memoria Virtual y Paginación:** Abstrae la memoria física y traslada dinámicamente páginas o bloques entre el disco y la RAM bajo demanda.

* **Políticas de Reemplazo y Ubicación:** Aplica algoritmos (como LRU o el algoritmo del Reloj) para decidir qué páginas expulsar de la RAM cuando la capacidad está colmada.

* **Control de Coherencia:** Se encarga de sincronizar las copias de información que han sido modificadas en la RAM (páginas o bloques "sucios") hacia el almacenamiento permanente.

* **Cachés de Software (** **Disk Cache** **/** **Buffer Cache** **):** Utiliza zonas de la RAM para almacenar bloques de archivos, estructuras del sistema de archivos y directorios accedidos frecuentemente, reduciendo las operaciones mecánicas de entrada/salida

---

## 3. Sección Práctica 1: Inspección de Recursos del Sistema (Bash / CLI)

En esta sección exploraremos el sistema sobre el cual se ejecuta este Cuaderno (Servidor Linux provisto por Google).

### [CÓDIGO 1.1] Identificación del Kernel y la Distribución Linux
```bash
# Ejecutamos comandos Bash usando el prefijo !
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

In [ ]:
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

=== INFORMACIÓN DEL KERNEL ===
Linux 759603cded87 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

=== DISTRIBUCIÓN DE LINUX ===
PRETTY_NAME="Ubuntu 22.04.5 LTS"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy


### [CÓDIGO 1.2] Inspección de la CPU (Recurso Procesador)
```bash
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

In [ ]:
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

=== ARQUITECTURA Y NÚCLEOS DE CPU ===
Architecture:                            x86_64
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
NUMA node0 CPU(s):                       0,1


### [CÓDIGO 1.3] Inspección de la Memoria RAM y Jerarquía de Almacenamiento
```bash
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

In [ ]:
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

=== ESTADO DE LA MEMORIA RAM (en MB) ===
               total        used        free      shared  buff/cache   available
Mem:           12975         750        8996           2        3227       11942
Swap:              0           0           0

=== SISTEMA DE ARCHIVOS Y DISCO DURA ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


### Pregunta de Análisis 1 (Responde en la celda de texto):
Basándote en las lecturas de Tanenbaum en NotebookLM y en los resultados obtenidos arriba:

1. ¿Qué versión de Kernel Linux está ejecutando la máquina virtual de Colab?  
**RESPUESTA:** 6.6.122+

2. ¿Cuántos núcleos de procesador y cuánta memoria RAM tiene asignados tu entorno virtual?  
**RESPUESTA:** nucleos de procesador 2  
memoria asignada 12975

## 4. Sección Práctica 2: Abstracción de Hardware a través del Sistema `/proc`

En Linux, `/proc` es un **pseudo-sistema de archivos**. No ocupa espacio en disco; es una interfaz directa mediante la cual el Kernel expone información sobre el estado de los recursos y procesos del sistema.

### [CÓDIGO 2.1] Exploración del sistema /proc
```bash
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

In [ ]:
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ===
MemTotal:       13286944 kB
MemFree:         9178160 kB
MemAvailable:   12208672 kB
Buffers:          130696 kB
Cached:          3097036 kB
SwapCached:            0 kB
Active:           477160 kB
Inactive:        3394316 kB
Active(anon):       1588 kB
Inactive(anon):   644492 kB

=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ===
965.59 1621.17


### [CÓDIGO 2.2] Visualización del árbol de llamadas e interrupciones del hardware
```bash
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

In [ ]:
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

=== INTERRUPCIONES DE HARDWARE REGISTRADAS ===
           CPU0       CPU1       
  0:        260          0   IO-APIC   0-edge      timer
  1:          0          9   IO-APIC   1-edge      i8042
  4:       1554       1214   IO-APIC   4-edge      ttyS0
  8:          0          0   IO-APIC   8-edge      rtc0
  9:          0          0   IO-APIC   9-fasteoi   acpi
 10:          0       1078   IO-APIC  10-fasteoi   virtio2
 12:          3          0   IO-APIC  12-edge      i8042
 24:          0          0  PCI-MSIX-0000:00:06.0   0-edge      virtio3-config
 25:          0         29  PCI-MSIX-0000:00:06.0   1-edge      virtio3-input
 26:          0          0  PCI-MSIX-0000:00:03.0   0-edge      virtio0-config
 27:          0          0  PCI-MSIX-0000:00:03.0   1-edge      virtio0-control
 28:          0          0  PCI-MSIX-0000:00:03.0   2-edge      virtio0-event
 29:          0     120142  PCI-MSIX-0000:00:03.0   3-edge      virtio0-request
 30:          0          0  PCI-MSIX-0000:00:0

## 5. Sección Práctica 3: Primer Contacto con Bajo Nivel (C / Syscalls POSIX)

A continuación, crearemos, compilaremos y ejecutaremos un programa en C para observar directamente las **Llamadas al Sistema** (`sys_write`, `sys_getpid`) que interactúan entre el **Modo Usuario** y el **Modo Kernel**.

```bash
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

```bash
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

In [1]:
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

# include <stdio.h>
# include <unistd.h>
# include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

Writing syscall_demo.c


### [CÓDIGO 3.2] Compilación y Ejecución del Código C usando GCC
```bash
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

In [ ]:
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 2703
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 2477
--------------------------------------------------


### [CÓDIGO 3.3] Rastreando las llamadas al sistema reales con `strace`
```bash
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

In [ ]:
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

getpid()                                = 3312
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
write(1, "[MODO USUARIO] Ejecutando progra"..., 43[MODO USUARIO] Ejecutando programa en C...
) = 43
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 50[LLAMADA AL SISTEMA] PID del Proceso Actual: 3312
) = 50
write(1, "[LLAMADA AL SISTEMA] PID del Pro"..., 63[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 3309
) = 63
write(1, "--------------------------------"..., 51--------------------------------------------------
) = 51
+++ exited with 0 +++


## 6. Desafío Asíncrono / Entregable de la Unidad I

### Instrucciones:
1. Modifica el archivo `syscall_demo.c` de la Sección 5 creando un nuevo archivo `desafio_u1.c` mediante celdas mágicas `%%writefile`.
2. Agrega la llamada al sistema POSIX `getuid()` (para obtener el ID del usuario que ejecuta el proceso) y la función `sleep(5)` para pausar la ejecución durante 5 segundos.
3. Ejecuta el programa en segundo plano mediante Bash y utiliza el comando `ps aux` para capturar el proceso en la lista de procesos activos mientras duerme.
4. Vuelve a ejecutar `strace` sobre el nuevo binario para identificar las syscalls `nanosleep` y `getuid`.

---

In [ ]:
%%writefile desafio_u1.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso y pausar la ejecución.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    pid_t process_id = getpid();
    pid_t parent_id = getppid();
    uid_t user_id = getuid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("[LLAMADA AL SISTEMA] UID del Usuario Actual: %u\n", user_id);
    printf("[LLAMADA AL SISTEMA] Pausando ejecución durante 5 segundos...\n");
    printf("--------------------------------------------------\n");

    sleep(5); // Pausar la ejecución durante 5 segundos (internamente usa nanosleep)

    printf("[MODO USUARIO] Proceso reanudado y finalizado.\n");

    return 0;
}

Overwriting desafio_u1.c


In [ ]:
!gcc desafio_u1.c -o desafio_u1
print("¡Compilación exitosa! Binario 'desafio_u1' listo.")

¡Compilación exitosa! Binario 'desafio_u1' listo.


In [ ]:
%%bash
# Ejecutar el binario en segundo plano y guardar su PID
./desafio_u1 &
BG_PID=$!

# Esperar 1 segundo para asegurar que el proceso esté en la fase de sleep
sleep 1

echo "--------------------------------------------------"
echo "[INSPECCIÓN DE PROCESOS] Buscando el proceso activo con ps aux:"
echo "--------------------------------------------------"
ps aux | grep desafio_u1 | grep -v grep

# Esperar a que el proceso en segundo plano termine sus 5 segundos
wait $BG_PID

--------------------------------------------------
[INSPECCIÓN DE PROCESOS] Buscando el proceso activo con ps aux:
--------------------------------------------------
root       11332  0.0  0.0   2684  1780 ?        S    23:23   0:00 ./desafio_u1
--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 11332
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 11331
[LLAMADA AL SISTEMA] UID del Usuario Actual: 0
[LLAMADA AL SISTEMA] Pausando ejecución durante 5 segundos...
--------------------------------------------------
[MODO USUARIO] Proceso reanudado y finalizado.


In [ ]:
%%bash
echo "--------------------------------------------------"
echo "[STRACE] Analizando llamadas al sistema de desafio_u1:"
echo "--------------------------------------------------"
strace ./desafio_u1 2>&1 | grep -E "getuid|nanosleep|exit"

--------------------------------------------------
[STRACE] Analizando llamadas al sistema de desafio_u1:
--------------------------------------------------
getuid()                                = 0
clock_nanosleep(CLOCK_REALTIME, 0, {tv_sec=5, tv_nsec=0}, 0x7ffe311fd610) = 0
exit_group(0)                           = ?
+++ exited with 0 +++


### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica1_Conceptos_SO.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.